In [21]:
import os
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

In [22]:
# Mount Google Drive (required every time)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
# Define and check the paths
# PROJECT_ROOT assumes the shared Milestone II folder is in your root google drive
PROJECT_ROOT = "/content/drive/MyDrive/Milestone II - NLP Cryptic Crossword Clues" # Sahana's Root Filepath
DATA_DIR = f"{PROJECT_ROOT}/data"
NOTEBOOK_DIR = f"{PROJECT_ROOT}/notebooks"

if not os.path.exists(PROJECT_ROOT):
    PROJECT_ROOT = os.path.abspath("..")  # fallback for local runs

In [24]:
# Read each CSV file into a DataFrame
df_clues = pd.read_csv(f'{DATA_DIR}/clues_raw.csv')
df_indicators = pd.read_csv(f'{DATA_DIR}/indicators_raw.csv')
df_ind_by_clue = pd.read_csv(f'{DATA_DIR}/indicators_by_clue_raw.csv')
df_ind_consolidated = pd.read_csv(f'{DATA_DIR}/indicators_consolidated_raw.csv')
df_charades = pd.read_csv(f'{DATA_DIR}/charades_raw.csv')
df_charades_by_clue = pd.read_csv(f'{DATA_DIR}/charades_by_clue_raw.csv')

## Filtering to clues with single word definitions and answers

In [25]:
df_clues.head()

,clue_id,clue,answer,definition,clue_number,puzzle_date,puzzle_name,source_url,source
0,1,"Acquisitive chap, as we see it (8)",COVETOUS,Acquisitive,1a,2019-08-08,Times 27424,https://times-xwd-times.livejournal.com/218581...,times_xwd_times
1,2,Back yard fencing weak and sagging (6),DROOPY,sagging,5a,2019-08-08,Times 27424,https://times-xwd-times.livejournal.com/218581...,times_xwd_times
2,3,"Stripping off uniform, love holding colonel's ...",UNCLOTHING,Stripping,8a,2019-08-08,Times 27424,https://times-xwd-times.livejournal.com/218581...,times_xwd_times
3,4,Without a mark where they should be gained (4),EXAM,where they should be gained,9a,2019-08-08,Times 27424,https://times-xwd-times.livejournal.com/218581...,times_xwd_times
4,5,"Put a stop to Rugby's foul school leader (5,2,...",KNOCK ON THE HEAD,Put a stop to,10a,2019-08-08,Times 27424,https://times-xwd-times.livejournal.com/218581...,times_xwd_times


In [26]:
df_clues.dtypes

,0
clue_id,int64
clue,object
answer,object
definition,object
clue_number,object
puzzle_date,object
puzzle_name,object
source_url,object
source,object


In [27]:
df_clues['answer_wc'] = df_clues['answer'].astype(str).apply(lambda x: len(x.split()))
df_clues['definition_wc'] = df_clues['definition'].astype(str).apply(lambda x: len(x.split()))

In [28]:
df_clues.head()

,clue_id,clue,answer,definition,clue_number,puzzle_date,puzzle_name,source_url,source,answer_wc,definition_wc
0,1,"Acquisitive chap, as we see it (8)",COVETOUS,Acquisitive,1a,2019-08-08,Times 27424,https://times-xwd-times.livejournal.com/218581...,times_xwd_times,1,1
1,2,Back yard fencing weak and sagging (6),DROOPY,sagging,5a,2019-08-08,Times 27424,https://times-xwd-times.livejournal.com/218581...,times_xwd_times,1,1
2,3,"Stripping off uniform, love holding colonel's ...",UNCLOTHING,Stripping,8a,2019-08-08,Times 27424,https://times-xwd-times.livejournal.com/218581...,times_xwd_times,1,1
3,4,Without a mark where they should be gained (4),EXAM,where they should be gained,9a,2019-08-08,Times 27424,https://times-xwd-times.livejournal.com/218581...,times_xwd_times,1,5
4,5,"Put a stop to Rugby's foul school leader (5,2,...",KNOCK ON THE HEAD,Put a stop to,10a,2019-08-08,Times 27424,https://times-xwd-times.livejournal.com/218581...,times_xwd_times,4,4


In [29]:
single_word_clues = df_clues[(df_clues['answer_wc'] == 1) & (df_clues['definition_wc'] == 1)].copy()

In [30]:
single_word_clues

,clue_id,clue,answer,definition,clue_number,puzzle_date,puzzle_name,source_url,source,answer_wc,definition_wc
0,1,"Acquisitive chap, as we see it (8)",COVETOUS,Acquisitive,1a,2019-08-08,Times 27424,https://times-xwd-times.livejournal.com/218581...,times_xwd_times,1,1
1,2,Back yard fencing weak and sagging (6),DROOPY,sagging,5a,2019-08-08,Times 27424,https://times-xwd-times.livejournal.com/218581...,times_xwd_times,1,1
2,3,"Stripping off uniform, love holding colonel's ...",UNCLOTHING,Stripping,8a,2019-08-08,Times 27424,https://times-xwd-times.livejournal.com/218581...,times_xwd_times,1,1
5,6,Foreign letter coming in is the French letter (7),EPISTLE,letter,11a,2019-08-08,Times 27424,https://times-xwd-times.livejournal.com/218581...,times_xwd_times,1,1
6,7,Charge to pack knick-knacks hurriedly (7),AGITATO,hurriedly,13a,2019-08-08,Times 27424,https://times-xwd-times.livejournal.com/218581...,times_xwd_times,1,1
...,...,...,...,...,...,...,...,...,...,...,...
660604,663644,Split peas cooked with speed (8),SEPARATE,Split,8d,2023-06-21,Daily Telegraph 30332,http://bigdave44.com/2023/06/21/dt-30332/,bigdave44,1,1
660608,663648,"You reportedly lead attack, getting reprimand (7)",UPBRAID,reprimand,18d,2023-06-21,Daily Telegraph 30332,http://bigdave44.com/2023/06/21/dt-30332/,bigdave44,1,1
660610,663650,Gather case of Sancerre will rise in value (6),ESTEEM,value,20d,2023-06-21,Daily Telegraph 30332,http://bigdave44.com/2023/06/21/dt-30332/,bigdave44,1,1
660611,663651,Open a place to drink in Paris? (5),UNBAR,Open,22d,2023-06-21,Daily Telegraph 30332,http://bigdave44.com/2023/06/21/dt-30332/,bigdave44,1,1


In [31]:
single_word_clues.isna().sum()/single_word_clues.shape[0]

,0
clue_id,0.000000
clue,0.000441
answer,0.005640
definition,0.357454
clue_number,0.001103
puzzle_date,0.048208
puzzle_name,0.000999
source_url,0.000000
source,0.000000
answer_wc,0.000000


ISSUE: ~36% of "single word" definitions are NaN. After dropping clues with NaN values in the definition, answer or the clue, we have 234407 clues.

In [32]:
single_word_clues = single_word_clues.dropna(subset=['definition', 'answer', 'clue'], inplace=False)

In [33]:
single_word_clues

,clue_id,clue,answer,definition,clue_number,puzzle_date,puzzle_name,source_url,source,answer_wc,definition_wc
0,1,"Acquisitive chap, as we see it (8)",COVETOUS,Acquisitive,1a,2019-08-08,Times 27424,https://times-xwd-times.livejournal.com/218581...,times_xwd_times,1,1
1,2,Back yard fencing weak and sagging (6),DROOPY,sagging,5a,2019-08-08,Times 27424,https://times-xwd-times.livejournal.com/218581...,times_xwd_times,1,1
2,3,"Stripping off uniform, love holding colonel's ...",UNCLOTHING,Stripping,8a,2019-08-08,Times 27424,https://times-xwd-times.livejournal.com/218581...,times_xwd_times,1,1
5,6,Foreign letter coming in is the French letter (7),EPISTLE,letter,11a,2019-08-08,Times 27424,https://times-xwd-times.livejournal.com/218581...,times_xwd_times,1,1
6,7,Charge to pack knick-knacks hurriedly (7),AGITATO,hurriedly,13a,2019-08-08,Times 27424,https://times-xwd-times.livejournal.com/218581...,times_xwd_times,1,1
...,...,...,...,...,...,...,...,...,...,...,...
660603,663643,"US university, without hesitation, showing exc...",MERIT,excellence,7d,2023-06-21,Daily Telegraph 30332,http://bigdave44.com/2023/06/21/dt-30332/,bigdave44,1,1
660604,663644,Split peas cooked with speed (8),SEPARATE,Split,8d,2023-06-21,Daily Telegraph 30332,http://bigdave44.com/2023/06/21/dt-30332/,bigdave44,1,1
660608,663648,"You reportedly lead attack, getting reprimand (7)",UPBRAID,reprimand,18d,2023-06-21,Daily Telegraph 30332,http://bigdave44.com/2023/06/21/dt-30332/,bigdave44,1,1
660610,663650,Gather case of Sancerre will rise in value (6),ESTEEM,value,20d,2023-06-21,Daily Telegraph 30332,http://bigdave44.com/2023/06/21/dt-30332/,bigdave44,1,1


## Joining with Indicators

In [35]:
df_ind_one_word = pd.read_csv(f'{DATA_DIR}/df_ind_one_word.csv')

In [48]:
df_ind_one_word.dtypes

,0
Unnamed: 0,int64
ind_id,int64
wordplay,object
indicator,object
clue_ids,object
num_clues,int64
ind_wc,int64
enchant_check,bool
zipf_score,float64
in_wordnet,bool


In [46]:
single_word_clues[single_word_clues['clue_id'] == 301220]['clue'].values[0]

'Standard jeans — slim gent seeking every other option (6)'

In [47]:
single_word_clues[single_word_clues['clue_id'] == 623961]['clue'].values[0]

'Crêpe in France, Breton one containing local milk, not Italian (6)'